In [3]:
import pandas as pd
import sqlite3

# Load the working dataframe again
df = pd.read_csv(r'C:\Users\ibrah\Downloads\Medicare Geographic Variation - by National, State & County\Medicare Geographic Variation - by National, State & County\2023\2014-2023 Medicare Fee-for-Service Geographic Variation Public Use File.csv')

# Filter to county 2023
county_2023 = df[(df['BENE_GEO_LVL'] == 'County') & (df['YEAR'] == 2023)]

print("Rows:", len(county_2023))
print("Ready")

C:\Users\ibrah\AppData\Local\Temp\ipykernel_15620\2733742351.py:5: DtypeWarning: Columns (223,224) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'C:\Users\ibrah\Downloads\Medicare Geographic Variation - by National, State & County\Medicare Geographic Variation - by National, State & County\2023\2014-2023 Medicare Fee-for-Service Geographic Variation Public Use File.csv')


Rows: 3198
Ready


In [4]:
# 1 — create a database connection (creates a file called cms_equity.db)
conn = sqlite3.connect('cms_equity.db')

# 2 — load our dataframe into a table called cms_county
county_2023.to_sql('cms_county', conn, if_exists='replace', index=False)

# 3 — confirm it worked
result = pd.read_sql('SELECT COUNT(*) as total_rows FROM cms_county', conn)
print(result)

   total_rows
0        3198


In [5]:
result = pd.read_sql("""
    SELECT BENE_GEO_DESC, BENE_DUAL_PCT, ER_VISITS_PER_1000_BENES, TOT_MDCR_STDZD_PYMT_PC
    FROM cms_county
    LIMIT 10
""", conn)
print(result)

             BENE_GEO_DESC BENE_DUAL_PCT ER_VISITS_PER_1000_BENES  \
0        AK-Aleutians East        0.2869                 180.3279   
1        AK-Aleutians West        0.3517                  89.6552   
2             AK-Anchorage        0.2271                 599.7099   
3                AK-Bethel        0.6338                  68.3453   
4           AK-Bristol Bay        0.2047                 188.9764   
5               AK-Chugach          0.25                 534.0909   
6          AK-Copper River        0.3539                 292.1348   
7                AK-Denali         0.109                 334.5865   
8            AK-Dillingham        0.4646                  63.6792   
9  AK-Fairbanks North Star        0.1721                 563.0686   

  TOT_MDCR_STDZD_PYMT_PC  
0                7913.45  
1                8125.52  
2                9192.93  
3                8116.22  
4               10405.53  
5               12756.78  
6                5332.91  
7                5916.78

In [8]:
result = pd.read_sql("""
    SELECT BENE_GEO_DESC, BENE_DUAL_PCT, ER_VISITS_PER_1000_BENES, TOT_MDCR_STDZD_PYMT_PC,BENE_AVG_RISK_SCRE, ACUTE_HOSP_READMSN_PCT
    FROM cms_county
    ORDER BY BENE_DUAL_PCT DESC
    LIMIT 20
""", conn)
print(result)

            BENE_GEO_DESC BENE_DUAL_PCT ER_VISITS_PER_1000_BENES  \
0             AK-Kusilvak        0.7018                  54.5809   
1               AK-Bethel        0.6338                  68.3453   
2                TX-Starr          0.58                 803.4464   
3             CA-Imperial        0.5764                 729.3022   
4     AK-Northwest Arctic         0.574                 172.1854   
5                 AK-Nome        0.5484                 632.2581   
6        SD-Oglala Lakota        0.5204                 576.5306   
7                 SD-Todd        0.5068                 441.0959   
8        CA-San Francisco         0.505                 603.8563   
9             KY-Franklin        0.5033                1019.4866   
10             SD-Buffalo        0.4727                 963.6364   
11          AK-Dillingham        0.4646                  63.6792   
12            TX-Maverick        0.4549                 656.6811   
13              TX-Zapata        0.4522         

In [14]:
result = pd.read_sql("""
    WITH min_max  AS(
    SELECT 
        MIN(CAST(BENE_DUAL_PCT AS FLOAT)) AS dual_min,
        MAX(CAST(BENE_DUAL_PCT AS FLOAT)) AS dual_max,
        MIN(CAST(BENE_AVG_RISK_SCRE AS FLOAT)) AS risk_min,
        MAX(CAST(BENE_AVG_RISK_SCRE AS FLOAT)) AS risk_max
    FROM cms_county
    WHERE BENE_DUAL_PCT != '*'
    AND BENE_AVG_RISK_SCRE != '*'
    ),
    normalized AS(
    SELECT
        c.BENE_GEO_DESC as geo_descr, 
        CAST( c.BENE_DUAL_PCT AS FLOAT) as dual_pct, 
        CAST( c.ER_VISITS_PER_1000_BENES AS FLOAT) as er_intensity, 
        CAST(c.TOT_MDCR_STDZD_PYMT_PC AS FLOAT) as total_spending,
        CAST(c.BENE_AVG_RISK_SCRE AS FLOAT) as risk_score, 
        CAST(c.ACUTE_HOSP_READMSN_PCT AS FLOAT) as readmission_rate,
        (CAST(c. BENE_DUAL_PCT AS FLOAT)- m.dual_min) / (m.dual_max - m.dual_min) AS dual_normalized,
        (CAST(c. BENE_AVG_RISK_SCRE AS FLOAT)- m.risk_min) / (m.risk_max - m.risk_min) AS risk_normalized
        
    FROM cms_county c, min_max m
    WHERE BENE_DUAL_PCT != '*'
    AND BENE_AVG_RISK_SCRE != '*'
    )
    SELECT
        geo_descr,
        dual_pct,
        er_intensity,
        total_spending,
        risk_score,
        readmission_rate,
        dual_normalized,
        risk_normalized,
        (dual_normalized + risk_normalized) as need_score
    FROM normalized
    ORDER BY need_score DESC
    LIMIT 20 
    
""", conn)
print(result)

           geo_descr  dual_pct  er_intensity  total_spending  risk_score  \
0   SD-Oglala Lakota    0.5204      576.5306        21798.68        1.74   
1           TX-Starr    0.5800      803.4464        17395.38        1.45   
2        CA-Imperial    0.5764      729.3022        14150.14        1.32   
3         SD-Buffalo    0.4727      963.6364        13460.70        1.41   
4            SD-Todd    0.5068      441.0959        17220.95        1.32   
5        KY-Franklin    0.5033     1019.4866        15294.68        1.19   
6           NY-Kings    0.4342      546.3319        14527.70        1.27   
7          TX-Zapata    0.4522      622.6115        13095.32        1.24   
8        TX-Jim Hogg    0.3827      956.7901        18045.82        1.35   
9         TX-Willacy    0.3808      604.2447        14296.38        1.35   
10          ND-Sioux    0.3786      390.9465        11980.38        1.35   
11       AK-Kusilvak    0.7018       54.5809         7960.16        0.78   
12     FL-Mi

In [24]:
result = pd.read_sql("""
    SELECT 
    BENE_GEO_DESC, 
    BENE_DUAL_PCT , 
    ER_VISITS_PER_1000_BENES, 
    TOT_MDCR_STDZD_PYMT_PC,
    BENE_AVG_RISK_SCRE, 
    ACUTE_HOSP_READMSN_PCT
    FROM cms_county
    WHERE
    TOT_MDCR_STDZD_PYMT_PC > (SELECT AVG(CAST(TOT_MDCR_STDZD_PYMT_PC AS FLOAT))FROM cms_county) 
    AND
    BENE_DUAL_PCT < (SELECT AVG(CAST(BENE_DUAL_PCT AS FLOAT))FROM cms_county)
    ORDER BY TOT_MDCR_STDZD_PYMT_PC DESC
    LIMIT 20
    """, conn)
print(result)

    BENE_GEO_DESC BENE_DUAL_PCT ER_VISITS_PER_1000_BENES  \
0      MT-Prairie        0.1244                 479.2746   
1   IL-Livingston        0.1214                 598.5417   
2      VA-Fairfax        0.0787                 467.6784   
3        MI-Emmet        0.1391                   539.71   
4         UT-Iron        0.0864                 551.8022   
5        ID-Power        0.1274                 593.4466   
6        KY-Scott        0.1027                 598.9247   
7       MO-Benton        0.1192                 485.2123   
8      VA-Augusta        0.0649                 584.7775   
9         NE-Loup             *                 411.7647   
10     IA-Mahaska        0.1044                 579.8289   
11     VA-Halifax        0.1387                 693.9558   
12      IL-Warren        0.1283                 482.2574   
13     SD-Douglas        0.0783                 332.7402   
14      NV-Washoe        0.1214                 562.6601   
15      IL-McLean        0.0878         

In [25]:
result = pd.read_sql("""
SELECT AVG(CAST(ACUTE_HOSP_READMSN_PCT AS FLOAT) )
FROM cms_county

""", conn)
print(result)

   AVG(CAST(ACUTE_HOSP_READMSN_PCT AS FLOAT) )
0                                     0.156516


In [26]:
result = pd.read_sql("""
    SELECT 
    BENE_GEO_DESC, 
    BENE_DUAL_PCT , 
    ER_VISITS_PER_1000_BENES, 
    TOT_MDCR_STDZD_PYMT_PC,
    BENE_AVG_RISK_SCRE, 
    ACUTE_HOSP_READMSN_PCT
    FROM cms_county
    WHERE
    TOT_MDCR_STDZD_PYMT_PC > (SELECT AVG(CAST(TOT_MDCR_STDZD_PYMT_PC AS FLOAT))FROM cms_county) 
    AND
    BENE_DUAL_PCT > (SELECT AVG(CAST(BENE_DUAL_PCT AS FLOAT))FROM cms_county)
    ORDER BY TOT_MDCR_STDZD_PYMT_PC DESC
    LIMIT 20
    """, conn)
print(result)

       BENE_GEO_DESC BENE_DUAL_PCT ER_VISITS_PER_1000_BENES  \
0         KY-Menifee        0.2598                    687.5   
1          AR-Searcy         0.206                 491.3858   
2          NM-Cibola        0.2651                 622.7945   
3           WV-Hardy        0.1882                 633.1331   
4          MT-Blaine        0.1981                 465.0718   
5          MN-Meeker        0.2165                 573.5057   
6            NC-Hyde        0.1558                 733.7662   
7      WV-Greenbrier        0.1682                 592.4927   
8       NM-Guadalupe        0.3218                 659.7701   
9          MO-Howell        0.1965                 625.0248   
10   NY-St. Lawrence        0.2037                 708.3533   
11  CA-San Francisco         0.505                 603.8563   
12       NY-Columbia        0.1782                 544.2185   
13       WI-Marathon        0.1921                 605.6182   
14           WI-Iron        0.2519                 579.

In [32]:
results = pd.read_sql(""" 
SELECT 
    BENE_GEO_DESC, 
    BENE_DUAL_PCT , 
    ER_VISITS_PER_1000_BENES, 
    TOT_MDCR_STDZD_PYMT_PC,
    BENE_AVG_RISK_SCRE, 
    ACUTE_HOSP_READMSN_PCT
FROM cms_county
ORDER BY CAST(ER_VISITS_PER_1000_BENES AS FLOAT) DESC
LIMIT 20
""", conn)
print(results)

         BENE_GEO_DESC BENE_DUAL_PCT ER_VISITS_PER_1000_BENES  \
0            IL-Hardin        0.2954                1535.3675   
1             KY-Wolfe        0.4097                1087.6565   
2          KY-Franklin        0.5033                1019.4866   
3             TX-Duval        0.3544                1012.0805   
4          MN-Mahnomen        0.2934                 998.2639   
5           TX-Refugio        0.1357                 985.5072   
6            TX-Reeves        0.2663                  984.862   
7        KY-Cumberland        0.3169                 964.4087   
8           SD-Buffalo        0.4727                 963.6364   
9            CA-Alpine        0.1614                 959.6413   
10         TX-Jim Hogg        0.3827                 956.7901   
11           NC-Bertie        0.2258                 951.4911   
12       LA-Washington        0.2853                 946.4373   
13        KY-Breathitt        0.4313                 937.1672   
14        SC-Allendale   

In [30]:
results = pd.read_sql(""" 
SELECT AVG(CAST(TOT_MDCR_STDZD_PYMT_PC AS FLOAT))
FROM cms_county
""", conn)
print(results)

   AVG(CAST(TOT_MDCR_STDZD_PYMT_PC AS FLOAT))
0                                11460.234891


In [31]:
results = pd.read_sql(""" 
SELECT AVG(CAST(ACUTE_HOSP_READMSN_PCT  AS FLOAT))
FROM cms_county
""", conn)
print(results)

   AVG(CAST(ACUTE_HOSP_READMSN_PCT  AS FLOAT))
0                                     0.156516
